# PI — architecture & live demo

Core: `pi/packages/{agent,coding-agent,ai}` (TypeScript, **plain async**). Condensed harness:
`pi/agent_harness/agent_harness` (Python). Write-up: `../01`, `../05`, `../06`.

## Architecture in one screen

- **Tiny model-agnostic loop:** `packages/agent/src/agent-loop.ts` — `runLoop`,
  `streamAssistantResponse`, `executeToolCalls`, `prepareToolCall`.

- **Harness:** `coding-agent/src/core/agent-session.ts` (`AgentSession`) wires persistence,
  extensions, compaction, retry; installs the `beforeToolCall`/`afterToolCall` hooks.

- **No built-in permission / MCP / sub-agents / plan mode** — all are extensions
  (`core/extensions/*`, ~30 hooks). This is PI's defining choice.

- **Persistence:** JSONL session *tree* (`session-manager.ts`), branch/fork/clone.

- **Steering vs follow-up queues** (`agent.ts:steer/followUp`).

- **35 provider catalogs** behind one `streamSimple` (`ai/src/compat.ts:265`).

In [ ]:
import os, sys

def find_repo_root():
    markers = {'mini-agent', 'pi', 'openhands_all'}
    d = os.path.abspath(os.getcwd())
    while True:
        try:
            if markers.issubset(set(os.listdir(d))):
                return d
        except OSError:
            pass
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    for cand in ('/repo', os.path.expanduser('~')):
        if os.path.isdir(os.path.join(cand, 'mini-agent')):
            return cand
    raise RuntimeError('repo root not found; set REPO manually')

REPO = find_repo_root()
print('REPO =', REPO)

In [ ]:
# Helper: print a slice of a REAL core source file around a symbol, so every
# architectural claim in this notebook can be checked against the implementation.
def show(relpath, needle=None, ctx=6, max_lines=40):
    path = os.path.join(REPO, relpath)
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(f'# {relpath}  ({len(lines)} lines)')
    if needle is None:
        start, end = 0, min(max_lines, len(lines))
    else:
        idx = next((i for i, l in enumerate(lines) if needle in l), None)
        if idx is None:
            print(f'  (needle {needle!r} not found)')
            return
        start, end = max(0, idx - 1), min(len(lines), idx + ctx)
    for i in range(start, end):
        print(f'{i+1:5} | {lines[i].rstrip()}')

### Verify against the real core

PI has **no built-in permission gate** — the only gate is the extension `tool_call` hook.
And there is **no MCP** in source. Check both:

In [ ]:
show('pi/packages/agent/src/agent-loop.ts', 'async function prepareToolCall', ctx=10)

In [ ]:
# 'No MCP by design' — there is no MCP client/server in pi source, only doc mentions.
import subprocess
res = subprocess.run(['grep', '-rIl', '--include=*.ts', 'ModelContextProtocol',
                      os.path.join(REPO, 'pi', 'packages')],
                     capture_output=True, text=True)
print('MCP client/server source files found:', res.stdout.strip() or '(none)')

## Live demo (condensed harness, offline)

PI's `agent_harness` runs the real turn shape with a scripted model: user → assistant(tool_call
`read_file`) → tool result → assistant(final). Persistence + tracing are pure EventBus
subscribers, exactly like `AgentSession._handleAgentEvent`.

In [ ]:
import tempfile
sys.path.insert(0, os.path.join(REPO, 'pi', 'agent_harness'))
from agent_harness import (Session, ScriptedModelProvider, PlannedResponse,
                           tool_call, default_tools, TraceRecorder)

workdir = tempfile.mkdtemp(prefix='pi_demo_')
open(os.path.join(workdir, 'notes.txt'), 'w').write('Ship on Friday. Owner: Mario.')

model = ScriptedModelProvider(plan=[
    PlannedResponse(tool_calls=[tool_call('read_file', {'path': 'notes.txt'})]),
    PlannedResponse(text='notes.txt says the release ships Friday, owned by Mario.'),
])
session = Session.create(model=model, cwd=workdir, tools=default_tools(workdir))
trace = TraceRecorder(); trace.attach(session.bus)
session.prompt('Summarize notes.txt')

print('TRANSCRIPT:')
for m in session.store.get_messages():
    print(f'  {m.role:12} {m.content[:55]!r}')
print('\nEVENT TRACE:')
for rec in trace.events:
    print('  ', rec)

**Why this is faithful to PI:** one `streamSimple` call per turn, tool results re-enter
history, the (optional) permission gate sits exactly where `beforeToolCall` does, and
persistence/tracing are event subscribers. The real PI adds compaction, retry, steering,
branching, and the extension API around this identical core.